[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C04_AI_Agents_Course/06_agentic_evals/06_agentic_evals.ipynb)

# 06 · Agentic 评测方法论 — pass@k / pass^k、time horizon、partial credit 与成本 Pareto

`CPU` · 全程离线可跑 · 配套讲解 [`06_讲解.html`](./06_讲解.html)

**本 notebook 的设计哲学**：我们用一个**参数化的模拟 agent** 把*评测方法论*与*模型能力*解耦——
所有指标管线（多 trial 统计、pass@k / pass^k、horizon 拟合、partial credit、成本 Pareto）对真实 agent
**原样适用**：把 `simulate_trial` 换成真正的 harness 调用（模块 03 的沙箱 + 模块 02 的 agent 循环）即可。
模拟的好处是：①不花一分钱 API 费；②每步成功率是我们*设定*的，因此每个估计量都有**已知真值可对照**——这是检验评测代码正确性的黄金方法。

流水线（对应讲解 §2 的 ASCII 图）：

1. 定义 6 个难度递增的 toy 任务，各带人类基线时长 `human_minutes` 与**确定性判分函数**；
2. 3 个"代际"的模拟 agent，每任务跑 N=20 trials → 成败矩阵；
3. pass@k 与 pass^k 估计 + 双曲线发散图 [Yao 2024, τ-bench]；
4. time-horizon 拟合：logistic(success ~ log2 t) → P50/P80 [Kwa 2025]；
5. partial credit：里程碑分 vs 二值分 [Wijk 2024, RE-Bench]；
6. 成功率-成本 Pareto 前沿 [Kapoor 2024]。

✏️ 3 道练习（无偏 pass^k 估计、Wilson 区间、bootstrap horizon CI），文末附参考答案。
论文引用见课程 [`references.md`](../references.md)。

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(20260608)   # 固定种子: 评测可复现的第一原则(讲解 §7)
np.set_printoptions(precision=3, suppress=True)
print("numpy", np.__version__)

## 1 · 任务套件与模拟 agent

**任务套件**：6 个任务难度递增，`human_minutes` ∈ {2, 5, 15, 40, 120, 300} 是"有相关技能的人类完成所需分钟数"
（horizon 方法论的"尺"，讲解 §2.2）。每个任务有确定性判分函数：**只看环境终态，不看 agent 自述**——
这里终态被抽象为 `steps_done`；真实任务族会检查文件存在/测试通过/数据库终态（模块 03/04、τ-bench 的做法）。

**模拟 agent**：任务需要 $S(t)=\max(4,\,\lceil 4\sqrt{t}\rceil)$ 步；agent 每步独立成功率
$q = 1-\varepsilon_0\,(1+0.55\log_2 t)$ —— **per-step 失败率随任务难度（人类时长）递增**，
且第一次出错即卡死（最悲观的 no-recovery 假设，对应讲解 §1 的误差复利模型 $P=q^S$）。
三个"代际" `agent-2023/2024/2025` 只差 $\varepsilon_0$（0.020 / 0.008 / 0.003），用来复现 horizon 逐代增长的故事。

每个 (代际 × 任务) 跑 **N=20 trials**，记录成败、完成步数与成本（步数 × 每步单价，新代际单价更高）。

In [ ]:
# ── 6 个 toy 任务: 难度递增, 各带人类基线时长(分钟) ──
TASKS = [
    {"name": "T1_config_fix", "human_minutes": 2,   "desc": "修复 JSON 配置中的一个字段错误"},
    {"name": "T2_unit_test",  "human_minutes": 5,   "desc": "修复一个失败的单元测试"},
    {"name": "T3_cross_bug",  "human_minutes": 15,  "desc": "定位并修复跨两个文件的逻辑 bug"},
    {"name": "T4_feature",    "human_minutes": 40,  "desc": "实现一个带边界条件的小功能"},
    {"name": "T5_refactor",   "human_minutes": 120, "desc": "重构模块并保持全部测试通过"},
    {"name": "T6_pipeline",   "human_minutes": 300, "desc": "搭建端到端数据管线并通过验证"},
]

def n_steps(human_minutes):
    # 任务所需 agent 步数: 随人类时长亚线性增长
    return max(4, int(round(4 * human_minutes ** 0.5)))

def grade_final_state(state):
    # 确定性判分: 只看环境终态。真实任务族在此检查文件/测试/数据库终态
    return state["steps_done"] >= state["steps_required"]

# ── 模拟 agent: 3 个代际, per-step 失败率 eps0*(1 + LAM*log2 t) 随难度递增 ──
GENERATIONS = {"agent-2023": 0.020, "agent-2024": 0.008, "agent-2025": 0.003}
LAM = 0.55
COST_PER_STEP = {"agent-2023": 0.004, "agent-2024": 0.012, "agent-2025": 0.050}  # $/步(模拟)

def p_step(eps0, human_minutes):
    return max(0.0, 1.0 - eps0 * (1.0 + LAM * math.log2(human_minutes)))

def simulate_trial(gen, task, rng):
    # 真实评测中, 这个函数 = harness 启动沙箱 → agent 循环 → 返回环境终态(模块 02/03)
    S = n_steps(task["human_minutes"])
    q = p_step(GENERATIONS[gen], task["human_minutes"])
    steps_done = 0
    for _ in range(S):
        if rng.random() < q:
            steps_done += 1
        else:
            break                          # 第一次出错即卡死: 最悲观的 no-recovery 假设
    state = {"steps_done": steps_done, "steps_required": S}
    return {"success": bool(grade_final_state(state)),
            "steps_done": steps_done,
            "cost": steps_done * COST_PER_STEP[gen]}

N_TRIALS = 20
results = {g: {t["name"]: [simulate_trial(g, t, rng) for _ in range(N_TRIALS)]
               for t in TASKS} for g in GENERATIONS}

# 成败矩阵一览 (成功数 c / N)
print(f"{'task':<14}{'t(min)':>7}{'steps':>7} | " + " | ".join(f"{g:>10}" for g in GENERATIONS))
for t in TASKS:
    cs = [sum(tr["success"] for tr in results[g][t["name"]]) for g in GENERATIONS]
    row = " | ".join(f"{c:>7}/{N_TRIALS}" for c in cs)
    print(f"{t['name']:<14}{t['human_minutes']:>7}{n_steps(t['human_minutes']):>7} | {row}")

## 2 · pass@k vs pass^k：能力视角与可靠性视角

设任务上单 trial 成功率为 $p$（trial 独立同分布）：

$$\text{pass@}k = 1-(1-p)^k \;\nearrow\; 1 \qquad\qquad \text{pass}^k = p^k \;\searrow\; 0 \qquad (k\uparrow)$$

- **pass@k**（k 次至少 1 次成功）：能力/elicitation 上界视角——"它*做得到*吗"；
- **pass^k**（k 次全部成功）：部署可靠性视角——"它*每次都做得到*吗"，τ-bench 把它推成 agent 评测头号指标 [Yao 2024]。

从 $n$ 次 trial、$c$ 次成功估计（无放回抽 $k$ 个 trial 的组合数技巧）：

$$\widehat{\text{pass@}k} = 1-\binom{n-c}{k}\Big/\binom{n}{k} \qquad\qquad \widehat{\text{pass}^k} = \binom{c}{k}\Big/\binom{n}{k}$$

下面先用 pass@k 的无偏估计 + pass^k 的朴素 plug-in $(c/n)^k$ 画发散图——
plug-in 是**有偏的（偏高）**，练习 1 让你实现无偏版并验证 $\binom{c}{k}/\binom{n}{k}\le (c/n)^k$。

In [ ]:
def pass_at_k(n, c, k):
    # 无偏估计: 1 - C(n-c,k)/C(n,k)  (k 次机会至少 1 次成功)
    if k > n:
        raise ValueError("k must be <= n")
    if n - c < k:
        return 1.0
    return 1.0 - math.comb(n - c, k) / math.comb(n, k)

def pass_pow_k_plugin(n, c, k):
    # 朴素 plug-in (c/n)^k —— 有偏(偏高), 练习 1 实现无偏版
    return (c / n) ** k

colors = {"agent-2023": "tab:red", "agent-2024": "tab:orange", "agent-2025": "tab:green"}
ks = np.arange(1, 11)
plt.figure(figsize=(7.5, 4.5))
for g in GENERATIONS:
    cs = {t["name"]: sum(tr["success"] for tr in results[g][t["name"]]) for t in TASKS}
    atk  = [np.mean([pass_at_k(N_TRIALS, cs[t["name"]], k) for t in TASKS]) for k in ks]
    powk = [np.mean([pass_pow_k_plugin(N_TRIALS, cs[t["name"]], k) for t in TASKS]) for k in ks]
    plt.plot(ks, atk,  "-o", color=colors[g], label=f"{g} pass@k")
    plt.plot(ks, powk, "--s", color=colors[g], alpha=0.55, label=f"{g} pass^k")
plt.xlabel("k"); plt.ylabel("mean over 6 tasks"); plt.ylim(0, 1.02)
plt.title("pass@k (solid, rises) vs pass^k (dashed, falls): divergence in k")
plt.legend(fontsize=8, ncol=2); plt.grid(alpha=0.3); plt.show()

# 同一系统, 两种叙事: 以 agent-2024 全套件为例
g = "agent-2024"
cs = {t["name"]: sum(tr["success"] for tr in results[g][t["name"]]) for t in TASKS}
a8 = np.mean([pass_at_k(N_TRIALS, cs[t["name"]], 8) for t in TASKS])
p8 = np.mean([pass_pow_k_plugin(N_TRIALS, cs[t["name"]], 8) for t in TASKS])
print(f"{g}:  pass@8 = {a8:.2f} ('它做得到')   vs   pass^8 = {p8:.2f} ('它每次都做得到')")

## ✏️ 练习 1：pass^k 的无偏估计 `pass_hat_k(n, c, k)`

实现 $\widehat{\text{pass}^k} = \binom{c}{k}/\binom{n}{k}$ —— 从 $n$ 个 trial 中无放回抽 $k$ 个、全部为成功的概率，
是 $p^k$ 的无偏估计（τ-bench 的官方口径 [Yao 2024]）。

**要求**：
1. `k > n` 时抛 `ValueError`（数据不够，估计无定义）；
2. `k > c` 时返回 `0.0`（成功 trial 不足 k 个）；
3. 其余返回 `math.comb(c, k) / math.comb(n, k)`。

**提示**：注意它与 plug-in 的关系 $\binom{c}{k}/\binom{n}{k} = \prod_{i=0}^{k-1}\frac{c-i}{n-i} \le (c/n)^k$，自测会检查这一不等式。

In [ ]:
def pass_hat_k(n, c, k):
    '''pass^k 的无偏估计 C(c,k)/C(n,k)。n=总 trial 数, c=成功数, k=连续成功要求次数'''
    # TODO: 1) k > n -> raise ValueError
    # TODO: 2) k > c -> return 0.0
    # TODO: 3) return math.comb(c, k) / math.comb(n, k)
    raise NotImplementedError("先完成 TODO")

In [ ]:
# ── 练习 1 自测 ──
assert abs(pass_hat_k(10, 5, 1) - 0.5) < 1e-12                    # k=1 退化为 c/n
assert abs(pass_hat_k(10, 5, 5) - 1 / math.comb(10, 5)) < 1e-12   # C(5,5)/C(10,5) = 1/252
assert pass_hat_k(20, 20, 7) == 1.0                               # 全成功
assert pass_hat_k(20, 0, 1) == 0.0                                # 全失败
assert pass_hat_k(10, 3, 5) == 0.0                                # k > c
try:
    pass_hat_k(5, 5, 6)
    raise AssertionError("k > n 应抛 ValueError")
except ValueError:
    pass
for n_, c_ in [(20, 12), (10, 7), (15, 3)]:
    prev = 1.1
    for k_ in range(1, 8):
        v = pass_hat_k(n_, c_, k_)
        assert v <= prev + 1e-12, "pass^k 应随 k 单调降"
        assert v <= (c_ / n_) ** k_ + 1e-12, "无偏估计应 <= plug-in"
        prev = v
print("✅ 练习 1 通过")

## 3 · Time horizon 拟合：logistic(success ~ log2 t) → P50 / P80

METR 的 time-horizon 方法论 [Kwa 2025]：把每个任务的人类基线时长 $t_i$ 当难度轴，对成败观测拟合

$$P(\text{success}\mid t) = \sigma\big(\beta_0 + \beta_1 \log_2 t\big),\ \ \beta_1<0
\qquad\Rightarrow\qquad
h_\alpha = 2^{(\operatorname{logit}\alpha - \beta_0)/\beta_1},\quad h_{50} = 2^{-\beta_0/\beta_1}$$

**P50 horizon** = 预测成功率为 50% 的人类时长；**P80**（部署语义更近）系统性短于 P50——
[Kwa 2025] 实测约为 P50 的 1/5。真实结果：2025 年初前沿模型 P50 ≈ 59 分钟，2019–2025 年间约每 7 个月翻倍。

下面用纯 numpy 梯度下降拟合（2 参数 logistic，无需 sklearn），对 3 个代际各拟合一条曲线，复现"horizon 逐代增长"。
注意两个讲解 §4 强调的陷阱在这里如何体现：horizon 是**该任务分布 + 该 scaffold**（这里是模拟器参数）的属性；
换任务集或换 scaffold，数字会平移。

In [ ]:
def fit_logistic(x, y, lr=2.0, iters=3000):
    # 2 参数 logistic 回归(梯度下降): 协变量标准化保证收敛, 再还原系数
    x = np.asarray(x, dtype=float); y = np.asarray(y, dtype=float)
    xm, xs = x.mean(), x.std()
    if xs == 0:
        xs = 1.0
    z = (x - xm) / xs
    w0 = w1 = 0.0
    for _ in range(iters):
        p = 1.0 / (1.0 + np.exp(-(w0 + w1 * z)))
        w0 -= lr * (p - y).mean()
        w1 -= lr * ((p - y) * z).mean()
    return w0 - w1 * xm / xs, w1 / xs        # (b0, b1) 还原到 log2(min) 坐标

def horizon_minutes(b0, b1, target=0.5):
    # 解 sigma(b0 + b1*log2 t) = target; 退化拟合截断到 [2^-5, 2^15] 分钟保持有限
    logit_t = math.log(target / (1.0 - target))
    if b1 >= 0:
        return 2.0 ** 15
    lg = (logit_t - b0) / b1
    return 2.0 ** min(max(lg, -5.0), 15.0)

fits = {}
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
t_grid = np.logspace(0, math.log10(600), 200)
for g in GENERATIONS:
    xs, ys = [], []
    for t in TASKS:
        for tr in results[g][t["name"]]:
            xs.append(math.log2(t["human_minutes"]))
            ys.append(1.0 if tr["success"] else 0.0)
    fits[g] = fit_logistic(xs, ys)
    emp = [np.mean([tr["success"] for tr in results[g][t["name"]]]) for t in TASKS]
    axes[0].scatter([t["human_minutes"] for t in TASKS], emp, color=colors[g], s=28)
    b0, b1 = fits[g]
    axes[0].plot(t_grid, 1 / (1 + np.exp(-(b0 + b1 * np.log2(t_grid)))), color=colors[g], label=g)
axes[0].axhline(0.5, ls=":", c="gray"); axes[0].set_xscale("log")
axes[0].set_xlabel("human time (min, log)"); axes[0].set_ylabel("success rate")
axes[0].set_title("logistic fit: success vs log2(human minutes)"); axes[0].legend(fontsize=8)

gen_names = list(GENERATIONS)
p50s = [horizon_minutes(*fits[g], 0.5) for g in gen_names]
p80s = [horizon_minutes(*fits[g], 0.8) for g in gen_names]
axes[1].plot(gen_names, p50s, "-o", label="P50 horizon")
axes[1].plot(gen_names, p80s, "-s", label="P80 horizon")
axes[1].set_yscale("log"); axes[1].set_ylabel("horizon (min, log)")
axes[1].set_title("horizon growth across generations")
axes[1].legend(); axes[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

for g in gen_names:
    print(f"{g}:  P50 = {horizon_minutes(*fits[g], 0.5):7.1f} min    P80 = {horizon_minutes(*fits[g], 0.8):6.1f} min")
print("→ P80 显著短于 P50: 把可靠性门槛从 50% 提到 80%, '能干的活'立刻缩水 (讲解 §4)")

## ✏️ 练习 2：pass@1 的 Wilson 置信区间 `wilson_ci_for_passk(c, n, z=1.96)`

单任务 n=20 个 trial 的成功率就是个二项比例——榜单上的"61%"没有区间就没有意义（讲解 §7 的方差陷阱）。
Wilson 区间比正态近似（Wald）在小样本/极端 $\hat p$ 下表现好得多：

$$\frac{\hat p + \frac{z^2}{2n} \;\pm\; z\sqrt{\frac{\hat p(1-\hat p)}{n} + \frac{z^2}{4n^2}}}{1 + z^2/n}$$

**要求**：返回 `(lo, hi)`，截断到 $[0,1]$。**提示**：$\hat p = c/n$；分母 `1 + z*z/n` 算一次复用；
留意 $c=0$ 时 Wilson 下界恰好为 0（Wald 会给出退化的 0 宽区间，这正是 Wilson 的优势）。

In [ ]:
def wilson_ci_for_passk(c, n, z=1.96):
    '''pass@1 = c/n 的 Wilson 置信区间, 返回 (lo, hi), 各自截断到 [0,1]'''
    # TODO: phat = c / n
    # TODO: denom = 1 + z*z/n
    # TODO: center = (phat + z*z/(2*n)) / denom
    # TODO: half = z * math.sqrt(phat*(1-phat)/n + z*z/(4*n*n)) / denom
    # TODO: return (max(0.0, center-half), min(1.0, center+half))
    raise NotImplementedError("先完成 TODO")

In [ ]:
# ── 练习 2 自测 ──
lo, hi = wilson_ci_for_passk(10, 20)
assert abs(lo + hi - 1.0) < 1e-9, "phat=0.5 时区间应关于 0.5 对称"
assert abs(lo - 0.2993) < 5e-4 and abs(hi - 0.7007) < 5e-4

lo0, hi0 = wilson_ci_for_passk(0, 20)
assert abs(lo0) < 1e-9 and 0.10 < hi0 < 0.20      # c=0: 下界恰为 0, 上界仍为正

lo1, hi1 = wilson_ci_for_passk(20, 20)
assert abs(hi1 - 1.0) < 1e-9 and 0.80 < lo1 < 0.90

for c_, n_ in [(3, 20), (15, 20)]:
    l, h = wilson_ci_for_passk(c_, n_)
    assert 0.0 <= l < c_ / n_ < h <= 1.0           # 包含点估计

w_small = np.subtract(*wilson_ci_for_passk(5, 10)[::-1])
w_large = np.subtract(*wilson_ci_for_passk(50, 100)[::-1])
assert w_large < w_small, "样本量增大区间应收窄"
print("✅ 练习 2 通过")

# 顺手看看真实数据: agent-2025 各任务的 pass@1 区间
for t in TASKS:
    c_ = sum(tr["success"] for tr in results["agent-2025"][t["name"]])
    l, h = wilson_ci_for_passk(c_, N_TRIALS)
    print(f"  agent-2025 {t['name']:<14} pass@1 = {c_/N_TRIALS:.2f}  Wilson95% = [{l:.2f}, {h:.2f}]")

## 4 · Partial credit：里程碑分 vs 二值分

二值分在长任务上信息量极低：$p\to 0$ 时，N=20 个 trial 经常给出 0/20——**两个弱系统不可区分**。
里程碑式评分把任务分解为 M 个**客观环境状态检查点**，得分 = 达成数/M，同样的 trial 预算下方差更小、对弱系统有区分度
（RE-Bench 更进一步用连续评分函数 $s_{\text{norm}}=(s-s_{\text{start}})/(s_{\text{human}}-s_{\text{start}})$ [Wijk 2024]）。

给最难任务 T6（300 min, 69 步）定义 3 个里程碑（**全部锚定环境状态、可自动判定**，绝不锚定轨迹文本观感——讲解 §5 的主观性纪律）：
- M1 环境与数据就绪（≥25% 步数）；M2 管线主体跑通（≥60%）；M3 端到端验证通过（=二值成功）。

注意：里程碑分与二值分**回答不同问题**——里程碑 0.67 ≠ 完成率 67%。它管区分度与诊断，结论仍要看结果分。

In [ ]:
task6 = TASKS[-1]
S6 = n_steps(task6["human_minutes"])
milestones = [math.ceil(0.25 * S6), math.ceil(0.60 * S6), S6]
print(f"T6_pipeline: 共 {S6} 步; 里程碑步数阈值 = {milestones}\n")

print(f"{'generation':<13}{'binary mean +/- SE':>22}{'milestone mean +/- SE':>26}")
for g in GENERATIONS:
    trials = results[g][task6["name"]]
    b = np.array([1.0 if tr["success"] else 0.0 for tr in trials])
    m = np.array([sum(tr["steps_done"] >= ms for ms in milestones) / 3 for tr in trials])
    se_b = b.std(ddof=1) / math.sqrt(len(b))
    se_m = m.std(ddof=1) / math.sqrt(len(m))
    print(f"{g:<13}{b.mean():>14.3f} +/- {se_b:.3f}{m.mean():>18.3f} +/- {se_m:.3f}")

print("\n→ 弱代际的二值分趋于 0/20(互相不可区分且 SE 无信息),")
print("  里程碑分把代际差距拉开, 信息量(均值/SE)明显更高 —— 这就是 partial credit 的统计价值")

## 5 · 成功率-成本 Pareto 前沿 [Kapoor 2024]

只报成功率的榜单会被"过度采样"刷爆：pass@k 随 k 单调升，而成本也随 k 单调升——**不报成本，k 就是免费的**。
[Kapoor 2024] 的处方：报 (成本, 成功率) 二维点、比较看 Pareto 前沿。

重试预算 $k$ 下（成功即停）：期望尝试次数 $=\frac{1-(1-p)^k}{p}$，期望成本 = 尝试次数 × 单次 trial 平均成本。
下面对 3 个代际 × 重试预算 {1,2,4,8} 画 12 个点。**注意看**：旧代际 × 大预算经常在"成功率/美元"上压过新代际 × 小预算——
但这要求任务有**廉价验证器**能从 k 次尝试里挑出对的那次；没有验证器时你拿到的不是 pass@k 而是 pass^k 味道的数字（讲解 §6）。

In [ ]:
BUDGETS = [1, 2, 4, 8]
points = []                                # (gen, k, mean_cost, mean_succ)
for g in GENERATIONS:
    for k in BUDGETS:
        costs, succs = [], []
        for t in TASKS:
            trials = results[g][t["name"]]
            c = sum(tr["success"] for tr in trials)
            phat = c / N_TRIALS
            succs.append(pass_at_k(N_TRIALS, c, k))
            mean_cost = float(np.mean([tr["cost"] for tr in trials]))
            exp_attempts = (1 - (1 - phat) ** k) / phat if phat > 0 else k
            costs.append(exp_attempts * mean_cost)
        points.append((g, k, float(np.mean(costs)), float(np.mean(succs))))

# Pareto 前沿: 按成本升序, 保留成功率 running max
front, best = [], -1.0
for g, k, cost, succ in sorted(points, key=lambda r: r[2]):
    if succ > best:
        front.append((cost, succ)); best = succ

plt.figure(figsize=(7.5, 4.5))
for g, k, cost, succ in points:
    plt.scatter(cost, succ, color=colors[g], s=20 + 14 * k, zorder=3)
    plt.annotate(f"k={k}", (cost, succ), fontsize=7, xytext=(4, 3), textcoords="offset points")
plt.plot([c for c, s in front], [s for c, s in front], "k--", lw=1, zorder=2)
handles = [plt.Line2D([0], [0], marker="o", ls="", color=colors[g], label=g) for g in GENERATIONS]
handles.append(plt.Line2D([0], [0], ls="--", color="k", label="Pareto frontier"))
plt.legend(handles=handles, fontsize=8)
plt.xscale("log"); plt.xlabel("expected $ per task (log)"); plt.ylabel("success rate (pass@k)")
plt.title("success-cost Pareto: generation x retry budget"); plt.grid(alpha=0.3)
plt.show()

for g, k, cost, succ in sorted(points, key=lambda r: r[2]):
    mark = "*" if (cost, succ) in front else " "
    print(f"{mark} {g:<12} k={k}:  cost/task = ${cost:6.2f}   success = {succ:.3f}")
print("\n* = Pareto 前沿上的配置。观察旧代际+重试与新代际单发的相对位置")

## ✏️ 练习 3：horizon 的 bootstrap 置信区间 `horizon_with_ci(tasks, gen_results, B=500, seed=0)`

horizon 点估计没有区间就和裸成功率一样裸奔。关键：方差的主要来源是**任务维度**（哪 6 个任务进了套件），
不是 trial 维度——所以对**任务重采样**（[Kwa 2025] 的做法），而不是对 trial 重采样。

**步骤**：
1. 内部函数 `fit_p50(task_indices)`：把这些任务（可重复）的全部 trial 摊平成 `(log2 human_minutes, 0/1)`，
   调上面的 `fit_logistic` + `horizon_minutes(b0, b1, 0.5)`；
2. 点估计 = `fit_p50(range(len(tasks)))`；
3. B 次：`idx = rng_b.integers(0, len(tasks), size=len(tasks))`（有放回），收集 `fit_p50(idx)`；
4. 返回 `(point, percentile 2.5, percentile 97.5)`。

**提示**：退化重采样（如全是简单任务）会让拟合发散——`horizon_minutes` 已截断到 $[2^{-5}, 2^{15}]$ 分钟，区间因此必有限。
自测用 B=300 跑约 10–30 秒。

In [ ]:
def horizon_with_ci(tasks, gen_results, B=500, seed=0):
    '''任务维度 bootstrap。gen_results: {task_name: [trial dict,...]} (单代际)。
    返回 (P50 点估计, ci_lo, ci_hi)'''
    rng_b = np.random.default_rng(seed)

    def fit_p50(task_indices):
        xs, ys = [], []
        # TODO: 对每个 i in task_indices: t = tasks[i];
        #       把 gen_results[t["name"]] 的每个 trial 摊平成
        #       xs.append(log2(t["human_minutes"])), ys.append(0/1)
        # TODO: b0, b1 = fit_logistic(xs, ys); return horizon_minutes(b0, b1, 0.5)
        raise NotImplementedError

    # TODO: point = fit_p50(range(len(tasks)))
    # TODO: boots = [fit_p50(rng_b.integers(0, len(tasks), size=len(tasks))) for _ in range(B)]
    # TODO: lo, hi = np.percentile(boots, [2.5, 97.5]); return point, float(lo), float(hi)
    raise NotImplementedError("先完成 TODO")

In [ ]:
# ── 练习 3 自测 (B=300, 约 10-30 秒) ──
p50, lo, hi = horizon_with_ci(TASKS, results["agent-2024"], B=300, seed=7)
assert np.isfinite(p50) and np.isfinite(lo) and np.isfinite(hi), "估计必须有限"
assert 0 < lo <= p50 <= hi, "区间必须包含点估计"
assert hi > lo, "区间必须有非零宽度"
assert hi - lo < 2.0 ** 15, "区间必须有限宽"
assert 5 < p50 < 150, "agent-2024 的 P50 应在任务时长范围的中段"
print(f"✅ 练习 3 通过   P50 = {p50:.1f} min,  95% CI = [{lo:.1f}, {hi:.1f}] min")
print("→ 只有 6 个任务时区间很宽: 这就是讲解 §1 '样本量天然小' 的直观代价")

---
## 📖 参考答案

先自己做，再对照。三题各一个 cell，运行后可回头重跑对应自测 cell 验证。

In [ ]:
# 参考答案 · 练习 1 (先自己做, 再对照)
def pass_hat_k(n, c, k):
    '''pass^k 的无偏估计 C(c,k)/C(n,k)'''
    if k > n:
        raise ValueError("k must be <= n")
    if k > c:
        return 0.0
    return math.comb(c, k) / math.comb(n, k)

# 无偏性 sanity check: 模拟真值 p=0.6, n=20, k=3 -> E[估计] 应接近 p^3 = 0.216
sim = np.random.default_rng(0).binomial(20, 0.6, size=20000)
est = np.mean([pass_hat_k(20, int(c_), 3) for c_ in sim])
print(f"E[pass_hat_k] = {est:.4f}  vs  真值 p^3 = {0.6**3:.4f}  (plug-in 均值偏高: "
      f"{np.mean([(c_/20)**3 for c_ in sim]):.4f})")

In [ ]:
# 参考答案 · 练习 2 (先自己做, 再对照)
def wilson_ci_for_passk(c, n, z=1.96):
    '''pass@1 = c/n 的 Wilson 置信区间'''
    phat = c / n
    denom = 1.0 + z * z / n
    center = (phat + z * z / (2 * n)) / denom
    half = z * math.sqrt(phat * (1 - phat) / n + z * z / (4 * n * n)) / denom
    return (max(0.0, center - half), min(1.0, center + half))

print(wilson_ci_for_passk(10, 20))   # (0.2993, 0.7007)

In [ ]:
# 参考答案 · 练习 3 (先自己做, 再对照)
def horizon_with_ci(tasks, gen_results, B=500, seed=0):
    '''任务维度 bootstrap -> (P50 点估计, ci_lo, ci_hi)'''
    rng_b = np.random.default_rng(seed)

    def fit_p50(task_indices):
        xs, ys = [], []
        for i in task_indices:
            t = tasks[i]
            for tr in gen_results[t["name"]]:
                xs.append(math.log2(t["human_minutes"]))
                ys.append(1.0 if tr["success"] else 0.0)
        b0, b1 = fit_logistic(xs, ys)
        return horizon_minutes(b0, b1, 0.5)

    point = fit_p50(range(len(tasks)))
    boots = [fit_p50(rng_b.integers(0, len(tasks), size=len(tasks))) for _ in range(B)]
    lo, hi = np.percentile(boots, [2.5, 97.5])
    return point, float(lo), float(hi)

for g in GENERATIONS:
    p50, lo, hi = horizon_with_ci(TASKS, results[g], B=300, seed=7)
    print(f"{g}:  P50 = {p50:7.1f} min   95% CI = [{lo:7.1f}, {hi:7.1f}] min")

## 小结

你在一个完全可控的模拟环境里跑通了 agentic 评测的全套统计管线：

| 你做了什么 | 对应方法论 |
|---|---|
| 6 任务 × 3 代际 × 20 trials 成败矩阵，确定性终态判分 | task family + 多 trial 报告（讲解 §1/§2） |
| pass@k 升、pass^k 降的发散双曲线 + 无偏估计 $\binom{c}{k}/\binom{n}{k}$ | 能力 vs 可靠性视角 [Yao 2024] |
| logistic(success ~ log2 t) → P50/P80，3 代际 horizon 增长 + bootstrap CI | time horizon [Kwa 2025] |
| T6 里程碑分 vs 二值分：弱代际从不可区分到可区分 | partial credit [Wijk 2024] |
| 代际 × 重试预算的成功率-成本散点与 Pareto 前沿 | 成本感知报告 [Kapoor 2024] |

把 `simulate_trial` 换成模块 03 的 harness 调用，这套管线就是一个真实 agent 评测后端。
最后记住讲解 §8 的纪律：这些数字支持的声明是"**该 scaffold、该任务分布、该口径下**测得 X"——
往"模型能 Y"走的每一步推断都要明示并打折（under-elicitation！）。

**下一站 → 模块 07 · 多智能体与编排**：当被测对象从单 agent 变成 orchestrator + subagents，
本章的指标怎么继续适用、又会多出哪些新的归因难题。

---
## 🎯 真实数据胶囊题：真实多步任务上的 pass^k 可靠性

agent 要连对**所有**步骤才算成功：pass^k = p^k（每步成功率 p，k 步）。这和 pass@k(至少一次)相反——它随步数暴跌。用真实 GSM8K 步数分布，算 agent 端到端成功率怎样随步数崩塌。

> 本模块新增的**真实数据**练习：自包含、用真实公开数据把本章方法跑一遍。先做 TODO，`assert` 全过即通关，文末有参考答案。

In [ ]:
import os, json, urllib.request, re
import numpy as np
CACHE=os.path.expanduser("~/.ai_agents_data"); os.makedirs(CACHE,exist_ok=True)
def _f(url,fn,headers=None):
    p=os.path.join(CACHE,fn)
    if not os.path.exists(p):
        req=urllib.request.Request(url, headers=headers or {})
        open(p,"wb").write(urllib.request.urlopen(req,timeout=40).read())
    return p
def gsm8k(n=300):
    p=_f("https://raw.githubusercontent.com/openai/grade-school-math/master/grade_school_math/data/test.jsonl","gsm8k_test.jsonl")
    return [json.loads(l) for l in open(p).read().splitlines()[:n]]
def gold(a): return a.split("####")[-1].strip().replace(",","")
def mbpp(n=100):
    p=_f("https://raw.githubusercontent.com/google-research/google-research/master/mbpp/mbpp.jsonl","mbpp.jsonl")
    return [json.loads(l) for l in open(p).read().splitlines()[:n]]

rows=gsm8k(300)
ks=np.array([max(1, r["answer"].count("<<")) for r in rows])
print(f"真实任务步数: 均值={ks.mean():.1f} 最大={ks.max()}")

**练习**：实现 `end_to_end_success(p_step, ks)`：每个任务成功概率 `p_step**k`，返回平均（这是 pass^k 口径）。验证：即便单步 95%，多步任务整体成功率也明显下降。

In [ ]:
def end_to_end_success(p_step, ks):
    # TODO: mean(p_step ** ks)
    raise NotImplementedError


In [ ]:
# 自测
s95=end_to_end_success(0.95, ks); s99=end_to_end_success(0.99, ks)
assert s99 > s95
assert s95 < 0.95, "多步连乘后整体 << 单步成功率"
# 长任务比短任务更易整体失败
long=ks[ks>=np.median(ks)]; short=ks[ks<np.median(ks)]
assert end_to_end_success(0.9,long) < end_to_end_success(0.9,short)
print(f"单步95% -> 端到端{s95:.2f}; 单步99% -> {s99:.2f} ✓ (agent 可靠性随步数暴跌)")


### 📖 参考答案

In [ ]:
def end_to_end_success(p_step, ks):
    return float(np.mean(p_step ** np.asarray(ks)))
print("✓ pass^k(全对) 解释了为什么长 horizon agent 任务这么难")